# Notebook 23 — Swarm Intelligence

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/23_swarm_intelligence.ipynb)
## Emergent Behavior from Simple Agents

Many **simple agents** with local interactions produce **emergent global behavior** — no central controller needed.

| Concept | Inspiration | Agent Behavior |
|---------|------------|----------------|
| Exploration | Ant foraging | Each agent explores independently |
| Sharing | Pheromone trails | Agents share discoveries with neighbors |
| Convergence | Bee waggle dance | Good ideas attract more attention |
| Diversity | Genetic diversity | Different starting points prevent groupthink |

**Prerequisites:** Notebooks 21–22 (orchestration, shared state).

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Swarm Patterns — Intelligence from Simplicity

Key principles from nature:
- **Decentralization** — no single agent controls the swarm
- **Local interaction** — each agent only sees its neighbors
- **Stigmergy** — indirect coordination through shared environment
- **Emergence** — complex behavior arises from simple rules

We'll build LLM-based swarm agents that:
1. Explore independently
2. Share discoveries with neighbors
3. Build on each other's ideas
4. Converge on the best solutions

In [ ]:
# Utility: extract key ideas from text
def extract_ideas(text: str) -> List[str]:
    """Extract distinct ideas from text."""
    messages = [
        {"role": "system", "content": "Extract distinct ideas from the text. Return each idea on a new line prefixed with '- '. Maximum 5 ideas. Be concise."},
        {"role": "user", "content": text}
    ]
    result = generate(messages, max_new_tokens=200, temperature=0.3, do_sample=True)
    ideas = [line.strip().lstrip("- ").strip() for line in result.split("\n") if line.strip().startswith("-")]
    return ideas[:5] if ideas else [result.strip()[:100]]


def rate_idea(idea: str, topic: str) -> float:
    """Rate an idea's relevance and quality (0-1)."""
    messages = [
        {"role": "system", "content": "Rate this idea on a scale of 1-10 for relevance and originality. Reply with ONLY a number."},
        {"role": "user", "content": f"Topic: {topic}\nIdea: {idea}"}
    ]
    result = generate(messages, max_new_tokens=10, temperature=0.1, do_sample=False)
    try:
        score = float(re.search(r'(\d+\.?\d*)', result).group(1))
        return min(score / 10.0, 1.0)
    except:
        return 0.5

print("✓ Utility functions ready: extract_ideas(), rate_idea()")

## 2. SimpleSwarmAgent — The Building Block

Each swarm agent is identical and simple:
- **explore(topic)** — generate ideas about a topic
- **share_discovery()** — broadcast best idea to neighbors
- **receive_discovery()** — learn from a neighbor's idea
- **build_on()** — extend an idea further

In [ ]:
class SimpleSwarmAgent:
    """A simple swarm agent that explores, shares, and builds on ideas."""

    def __init__(self, agent_id: int, persona: str = ""):
        self.agent_id = agent_id
        self.persona = persona or f"creative thinker #{agent_id}"
        self.ideas: List[Dict[str, Any]] = []
        self.received: List[Dict[str, Any]] = []
        self.best_idea: Optional[Dict[str, Any]] = None
        self.generation_count = 0

    def explore(self, topic: str, angle: str = "") -> List[str]:
        """Generate new ideas about a topic."""
        self.generation_count += 1
        prompt = f"Brainstorm 2-3 creative and specific ideas about: {topic}"
        if angle:
            prompt += f"\nFocus on this angle: {angle}"
        if self.received:
            recent = self.received[-2:]
            context = "\n".join([f"- {r['idea']}" for r in recent])
            prompt += f"\nBuild on (but don't repeat) these related ideas:\n{context}"

        messages = [
            {"role": "system", "content": f"You are {self.persona}. Generate novel, specific ideas. Each idea should be 1-2 sentences. Be creative and practical."},
            {"role": "user", "content": prompt}
        ]
        response = generate(messages, max_new_tokens=200)
        new_ideas = extract_ideas(response)

        for idea in new_ideas:
            entry = {"idea": idea, "source": f"agent_{self.agent_id}", "gen": self.generation_count}
            self.ideas.append(entry)
            if self.best_idea is None or len(idea) > len(self.best_idea.get("idea", "")):
                self.best_idea = entry

        return new_ideas

    def share_discovery(self) -> Optional[Dict[str, Any]]:
        """Share the best idea with neighbors."""
        return self.best_idea

    def receive_discovery(self, discovery: Dict[str, Any]):
        """Receive and store a discovery from a neighbor."""
        if discovery and discovery["source"] != f"agent_{self.agent_id}":
            self.received.append(discovery)

    def build_on(self, idea: str, topic: str) -> str:
        """Extend or improve an existing idea."""
        messages = [
            {"role": "system", "content": f"You are {self.persona}. Take this idea and improve it or extend it in a novel direction. Be specific and concise (2-3 sentences)."},
            {"role": "user", "content": f"Topic: {topic}\nOriginal idea: {idea}\nYour improved/extended version:"}
        ]
        result = generate(messages, max_new_tokens=150)
        improved = {"idea": result.strip(), "source": f"agent_{self.agent_id}", "gen": self.generation_count, "built_on": idea[:50]}
        self.ideas.append(improved)
        return result.strip()

    def __repr__(self):
        return f"SwarmAgent({self.agent_id}, ideas={len(self.ideas)}, received={len(self.received)})"

# Test a single agent
test_agent = SimpleSwarmAgent(0, "innovative technologist")
ideas = test_agent.explore("Using AI to improve education")
print(f"Agent 0 explored and found {len(ideas)} ideas:")
for idea in ideas:
    print(f"  - {idea[:100]}")

## 3. SwarmCoordinator — Managing the Swarm

The coordinator manages N agents with:
- **Local communication** — each agent sees only k neighbors (ring topology)
- **Rounds** — explore → share → receive → explore again
- **Global best tracking** — track the swarm's overall best ideas

In [ ]:
class SwarmCoordinator:
    """Manages a swarm of agents with local communication topology."""

    def __init__(self, n_agents: int, k_neighbors: int = 2):
        self.agents = [SimpleSwarmAgent(i, f"specialist #{i}") for i in range(n_agents)]
        self.k = min(k_neighbors, n_agents - 1)
        self.global_best: List[Dict[str, Any]] = []
        self.round_log: List[Dict] = []
        self.all_ideas: List[Dict[str, Any]] = []

    def _get_neighbors(self, agent_idx: int) -> List[int]:
        """Ring topology: each agent connects to k nearest neighbors."""
        n = len(self.agents)
        neighbors = []
        for offset in range(1, self.k + 1):
            neighbors.append((agent_idx + offset) % n)
            neighbors.append((agent_idx - offset) % n)
        return list(set(neighbors))[:self.k]

    def run_round(self, topic: str, round_num: int, angles: List[str] = None) -> Dict:
        """Execute one round: explore → share → receive."""
        round_ideas = []
        print(f"\n--- Round {round_num} ---")

        # Phase 1: Explore
        for i, agent in enumerate(self.agents):
            angle = angles[i] if angles and i < len(angles) else ""
            ideas = agent.explore(topic, angle)
            round_ideas.extend(ideas)
            print(f"  Agent {i}: explored → {len(ideas)} ideas")

        # Phase 2: Share & Receive (local topology)
        for i, agent in enumerate(self.agents):
            discovery = agent.share_discovery()
            if discovery:
                for neighbor_idx in self._get_neighbors(i):
                    self.agents[neighbor_idx].receive_discovery(discovery)

        # Track all ideas
        for agent in self.agents:
            self.all_ideas.extend(agent.ideas[-len(round_ideas):])

        round_info = {
            "round": round_num,
            "total_ideas": len(round_ideas),
            "unique_ideas": len(set(round_ideas)),
        }
        self.round_log.append(round_info)
        return round_info

    def run_swarm(self, topic: str, n_rounds: int, angles: List[str] = None) -> Dict:
        """Run the full swarm for n rounds."""
        t0 = time.time()
        print(f"Swarm: {len(self.agents)} agents, {n_rounds} rounds, k={self.k} neighbors")
        print(f"Topic: {topic}")

        for r in range(n_rounds):
            self.run_round(topic, r + 1, angles)

        # Collect all unique ideas
        all_idea_texts = []
        for agent in self.agents:
            for entry in agent.ideas:
                all_idea_texts.append(entry["idea"])

        unique_ideas = list(set(all_idea_texts))
        elapsed = time.time() - t0

        return {
            "total_ideas": len(all_idea_texts),
            "unique_ideas": len(unique_ideas),
            "ideas_per_agent": {f"agent_{a.agent_id}": len(a.ideas) for a in self.agents},
            "time": round(elapsed, 2),
            "rounds": self.round_log,
            "all_unique": unique_ideas
        }

print("✓ SwarmCoordinator defined")

## 4. Experiment 1 — Brainstorming Swarm

5 agents brainstorm ideas on a topic. Over multiple rounds, they share discoveries with neighbors and build on each other's ideas. We track **idea diversity** and **convergence**.

In [ ]:
# Experiment 1: Brainstorming swarm
topic = "Innovative ways to reduce food waste using technology"

swarm = SwarmCoordinator(n_agents=5, k_neighbors=2)

# Different angles for each agent
angles = [
    "consumer/household level solutions",
    "supply chain and logistics",
    "AI and data analytics approaches",
    "community and social platforms",
    "hardware and IoT sensors"
]

print("=" * 70)
print("EXPERIMENT 1: BRAINSTORMING SWARM")
print("=" * 70)

result = swarm.run_swarm(topic, n_rounds=2, angles=angles)

print(f"\n{'=' * 70}")
print("RESULTS")
print("=" * 70)
print(f"Total ideas generated: {result['total_ideas']}")
print(f"Unique ideas: {result['unique_ideas']}")
print(f"Time: {result['time']}s")
print(f"\nIdeas per agent:")
for agent_id, count in result['ideas_per_agent'].items():
    print(f"  {agent_id}: {count} ideas")

In [ ]:
# Display the best ideas from the swarm
print("=" * 70)
print("ALL UNIQUE IDEAS FROM SWARM")
print("=" * 70)

for i, idea in enumerate(result['all_unique'][:15], 1):
    print(f"\n{i}. {idea[:200]}")

## 5. Experiment 2 — Research Exploration Swarm

5 agents each explore a **different aspect** of a research question, then share findings. This tests the swarm's ability to cover broad ground.

In [ ]:
# Experiment 2: Research exploration
research_topic = "How will quantum computing change cybersecurity in the next decade?"

research_swarm = SwarmCoordinator(n_agents=5, k_neighbors=2)
research_angles = [
    "current quantum computing capabilities and timeline",
    "vulnerabilities in current cryptographic systems",
    "post-quantum cryptography solutions",
    "impact on financial and government security",
    "quantum key distribution and quantum networks"
]

print("=" * 70)
print("EXPERIMENT 2: RESEARCH EXPLORATION SWARM")
print("=" * 70)

research_result = research_swarm.run_swarm(research_topic, n_rounds=2, angles=research_angles)

print(f"\n{'=' * 70}")
print("RESEARCH FINDINGS")
print("=" * 70)
print(f"Total findings: {research_result['total_ideas']}")
print(f"Unique findings: {research_result['unique_ideas']}")

# Group findings by agent
for agent in research_swarm.agents:
    print(f"\n[Agent {agent.agent_id} — {research_angles[agent.agent_id]}]:")
    for entry in agent.ideas[:3]:
        print(f"  • {entry['idea'][:150]}")

## 6. Diversity Analysis — Swarm vs Single Agent

Does a swarm produce **more diverse** ideas than a single powerful agent? Let's measure.

In [ ]:
# Single agent baseline — generate the same number of ideas
single_topic = "Innovative applications of AI in healthcare"

print("=" * 70)
print("DIVERSITY ANALYSIS: SWARM vs SINGLE AGENT")
print("=" * 70)

# Single agent: ask for many ideas at once
print("\n--- Single Agent (one powerful prompt) ---")
single_messages = [
    {"role": "system", "content": "You are a creative innovation expert. Generate as many diverse, specific ideas as possible."},
    {"role": "user", "content": f"Generate 10 innovative and specific ideas about: {single_topic}. Each idea should be 1-2 sentences. Number them."}
]
t0 = time.time()
single_response = generate(single_messages, max_new_tokens=500)
single_time = time.time() - t0
single_ideas = [line.strip() for line in single_response.split("\n") if line.strip() and any(c.isalpha() for c in line)]
print(f"  Generated {len(single_ideas)} ideas in {single_time:.1f}s")

# Swarm: 5 agents exploring
print("\n--- Swarm (5 agents × 1 round) ---")
diversity_swarm = SwarmCoordinator(n_agents=5, k_neighbors=2)
diversity_angles = [
    "diagnosis and imaging",
    "drug discovery and genomics",
    "patient monitoring and wearables",
    "mental health and therapy",
    "hospital operations and logistics"
]
t0 = time.time()
swarm_result = diversity_swarm.run_swarm(single_topic, n_rounds=1, angles=diversity_angles)
swarm_time = time.time() - t0

print(f"\n{'=' * 70}")
print("COMPARISON")
print("=" * 70)
print(f"""
                    │ Single Agent  │ Swarm (5 agents)
────────────────────┼───────────────┼──────────────────
Total ideas         │ {len(single_ideas):>11}   │ {swarm_result['total_ideas']:>14}
Unique ideas        │ {len(set(single_ideas)):>11}   │ {swarm_result['unique_ideas']:>14}
Time                │ {single_time:>10.1f}s  │ {swarm_time:>13.1f}s
Angles covered      │ {'1':>11}   │ {len(diversity_angles):>14}
""")

In [ ]:
# Show ideas side by side
print("SINGLE AGENT IDEAS:")
for i, idea in enumerate(single_ideas[:8], 1):
    print(f"  {i}. {idea[:120]}")

print(f"\nSWARM IDEAS (sample):")
for i, idea in enumerate(swarm_result['all_unique'][:8], 1):
    print(f"  {i}. {idea[:120]}")

## 7. Scaling — 3 vs 5 vs 7 Agents

How does swarm size affect idea diversity and cost? We test three configurations.

In [ ]:
# Scaling experiment
scaling_topic = "Creative uses of augmented reality in education"

scaling_results = {}
for n_agents in [3, 5, 7]:
    print(f"\n--- Testing {n_agents} agents ---")
    s = SwarmCoordinator(n_agents=n_agents, k_neighbors=2)
    t0 = time.time()
    r = s.run_swarm(scaling_topic, n_rounds=1)
    elapsed = time.time() - t0
    scaling_results[n_agents] = {
        "total": r["total_ideas"],
        "unique": r["unique_ideas"],
        "time": round(elapsed, 2)
    }

print(f"\n{'=' * 70}")
print("SCALING RESULTS")
print("=" * 70)
print(f"{'Agents':<10} {'Total Ideas':<15} {'Unique Ideas':<15} {'Time (s)':<12} {'Ideas/sec':<10}")
print("-" * 62)
for n, data in scaling_results.items():
    ips = data['unique'] / max(data['time'], 0.1)
    print(f"{n:<10} {data['total']:<15} {data['unique']:<15} {data['time']:<12} {ips:<10.2f}")

print(f"""
Observations:
  - More agents → more total ideas (roughly linear)
  - Unique ideas grow sub-linearly (some overlap)
  - Time grows linearly (sequential execution)
  - Sweet spot depends on diversity needs vs. budget
""")

## 8. Swarm vs Single Powerful Agent — When to Use Which

The swarm pattern isn't always better. Let's compare on a task that requires **depth** vs **breadth**.

In [ ]:
# Deep task: better for single focused agent
# Broad task: better for swarm

deep_task = "Explain the mathematical foundations of attention mechanisms in transformers"
broad_task = "List innovative applications of AI across different industries"

print("=" * 70)
print("SWARM vs SINGLE AGENT — DEPTH vs BREADTH")
print("=" * 70)

for task_type, task in [("DEEP", deep_task), ("BROAD", broad_task)]:
    print(f"\n{'─' * 60}")
    print(f"Task type: {task_type}")
    print(f"Task: {task[:70]}...")

    # Single agent
    t0 = time.time()
    single_msg = [
        {"role": "system", "content": "You are a knowledgeable expert. Provide a thorough response."},
        {"role": "user", "content": task}
    ]
    single_resp = generate(single_msg, max_new_tokens=400)
    single_time = time.time() - t0
    single_words = len(single_resp.split())

    # Swarm
    t0 = time.time()
    mini_swarm = SwarmCoordinator(n_agents=3, k_neighbors=1)
    swarm_r = mini_swarm.run_swarm(task, n_rounds=1)
    swarm_time = time.time() - t0

    print(f"\n  Single agent: {single_words} words, {single_time:.1f}s")
    print(f"  Swarm (3 agents): {swarm_r['unique_ideas']} unique ideas, {swarm_time:.1f}s")
    print(f"  Verdict: {'Single agent better (needs depth)' if task_type == 'DEEP' else 'Swarm better (needs breadth)'}")

## 9. Key Takeaways

### Swarm Intelligence Principles
1. **Simple agents + local rules = emergent intelligence** — no central controller needed
2. **Diversity** is the swarm's superpower — different angles prevent groupthink
3. **Local communication** (ring topology) prevents premature convergence
4. **Multiple rounds** let agents build on each other's discoveries

### When to Use Swarms
| Use Swarm | Use Single Agent |
|-----------|-----------------|
| Breadth: many ideas needed | Depth: detailed analysis |
| Exploration of unknown space | Well-defined problems |
| Brainstorming & creativity | Logical reasoning chains |
| Diverse perspectives needed | Consistency required |

### Scaling Observations
- Total ideas scale **linearly** with agent count
- Unique ideas scale **sub-linearly** (diminishing returns)
- Cost scales linearly — budget constraints matter
- Sweet spot: **3–7 agents** for most brainstorming tasks

### Design Tips
- Give each agent a **different angle** / persona for maximum diversity
- Use **2-3 rounds** — more rounds have diminishing returns
- **k=2 neighbors** balances exploration vs. convergence
- Aggregate results with a separate synthesizer agent

### Next Steps
→ **Notebook 24**: Agent safety and guardrails
→ **Notebook 25**: Human-in-the-loop agent systems